# Charlotte Coffee Shop Map

This notebook reads the coffee shop information from `coffees.xlsx` and creates the interactive Charlotte coffee shop map.

**To update the map:** edit the Excel file, save it, then run the notebook again. No coffee shop information needs to be edited in the Python code.

In [1]:
%pip install folium pandas openpyxl

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [23]:
import pandas as pd
import folium
from html import escape
from pathlib import Path

# ---------------------------------------------------------
# 1. Excel file
# ---------------------------------------------------------

excel_file = Path("C:/Users/gbartel/Downloads/coffees.xlsx")

df = pd.read_excel(excel_file)

# Clean up Excel headers
df.columns = [str(c).strip() for c in df.columns]

# Required columns
required = [
    "Coffee shop",
    "Latitude",
    "Longitude",
    "Dog friendly",
    "Remote work friendly"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required Excel columns: {missing}")

# Only map rows that have a name and coordinates
df = df.dropna(
    subset=["Coffee shop", "Latitude", "Longitude"]
).copy()

print(f"Loaded {len(df)} coffee shops.")


# ---------------------------------------------------------
# 2. Clean up Yes/No columns
# ---------------------------------------------------------
# This makes YES, Yes, yes, etc. all work the same way.

df["Dog friendly"] = (
    df["Dog friendly"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

df["Remote work friendly"] = (
    df["Remote work friendly"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)


# ---------------------------------------------------------
# 3. Create the map
# ---------------------------------------------------------

# Create map
m = folium.Map(
    location=[35.2271, -80.8431],
    zoom_start=13,
    tiles=None
)

folium.TileLayer(
    "https://tiles.stadiamaps.com/tiles/stamen_watercolor/{z}/{x}/{y}.jpg",
    attr='&copy; <a href="https://stadiamaps.com/">Stadia Maps</a>, '
         '&copy; <a href="https://openmaptiles.org">OpenMapTiles</a> '
         '&copy; <a href="http://openstreetmap.org">OpenStreetMap</a> contributors',
    name="Stamen Watercolor",
    max_zoom=20,
).add_to(m)
# Stamen Terrain labels on top
folium.TileLayer(
    "https://tiles.stadiamaps.com/tiles/stamen_terrain_labels/{z}/{x}/{y}@2x.png",
    attr='&copy; <a href="https://stadiamaps.com/">Stadia Maps</a>, '
         '&copy; <a href="https://openmaptiles.org">OpenMapTiles</a> '
         '&copy; <a href="http://openstreetmap.org">OpenStreetMap</a> contributors',
    name="Stamen Terrain Labels",
    overlay=True,
    control=True,
    show=True,
    max_zoom=20,
).add_to(m)



# ---------------------------------------------------------
# 4. Create the toggleable layers
# ---------------------------------------------------------

# Layer 1: All coffee shops
all_layer = folium.FeatureGroup(
    name="☕ All Coffee Shops",
    show=True
)

# Layer 2: Dog friendly
dog_layer = folium.FeatureGroup(
    name="🐾 Dog Friendly",
    show=False
)

# Layer 3: Remote work friendly
remote_layer = folium.FeatureGroup(
    name="💻 Remote Work Friendly",
    show=False
)


# ---------------------------------------------------------
# 5. Map title
# ---------------------------------------------------------
title_html = """
<div style="
    position: fixed;
    bottom: 20px;
    left: 50%;
    transform: translateX(-50%);
    z-index: 9999;
    background-color: white;
    padding: 10px 16px;
    border-radius: 8px;
    font-size: 18px;
    font-weight: bold;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
">
    The Charlotte Coffee Shop Map ☕
</div>
"""

m.get_root().html.add_child(
    folium.Element(title_html)
)


# ---------------------------------------------------------
# 6. Add coffee shops to the appropriate layers
# ---------------------------------------------------------
for _, row in df.iterrows():

    name = escape(str(row["Coffee shop"]))
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])


    # =====================================================
    # ALL COFFEE SHOPS LAYER
    # =====================================================

    # Coffee cup marker
    folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(
            icon="coffee",
            prefix="fa",
            color="lightblue",
            icon_color="white"
        ),
        tooltip=name
    ).add_to(all_layer)


    # Coffee shop name underneath the marker
    label_html = f"""
    <div style="
        width: 180px;
        margin-left: -90px;
        margin-top: -2px;
        text-align: center;
        white-space: normal;
        font-family: 'Trebuchet MS', sans-serif;
        font-size: 14px;
        line-height: 14px;
        font-weight: 600;
        color: #222;
        text-shadow:
            -1px -1px 0 #fff,
             1px -1px 0 #fff,
            -1px  1px 0 #fff,
             1px  1px 0 #fff;
        pointer-events: none;
    ">
        {name}
    </div>
    """

    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=label_html,
            icon_size=(180, 30),
            icon_anchor=(0, 0)
        )
    ).add_to(all_layer)


    # =====================================================
    # DOG FRIENDLY LAYER
    # =====================================================

    if row["Dog friendly"] == "yes":

        folium.Marker(
            location=[lat, lon],
            tooltip=f"{name} — Dog Friendly",
            icon=folium.Icon(
                icon="paw",
                prefix="fa",
                color="purple",
                icon_color="white"
            )
        ).add_to(dog_layer)


    # =====================================================
    # REMOTE WORK FRIENDLY LAYER
    # =====================================================

    if row["Remote work friendly"] == "yes":

        folium.Marker(
            location=[lat, lon],
            tooltip=f"{name} — Remote Work Friendly",
            icon=folium.Icon(
                icon="laptop",
                prefix="fa",
                color="gray",
                icon_color="white"
            )
        ).add_to(remote_layer)


# ---------------------------------------------------------
# 7. Add the layers to the map
# ---------------------------------------------------------

all_layer.add_to(m)
dog_layer.add_to(m)
remote_layer.add_to(m)


# ---------------------------------------------------------
# 8. Add the layer toggle control
# ---------------------------------------------------------

folium.LayerControl(
    collapsed=False,
    position="topright"
).add_to(m)


# ---------------------------------------------------------
# 9. Display the map
# ---------------------------------------------------------

m


Loaded 42 coffee shops.


In [25]:
# Save the finished map as an HTML file
output_file = "Charlotte_Coffee.html"
m.save(output_file)
print(f"Saved: {output_file}")


Saved: Charlotte_Coffee.html
